## 1. hwp to faiss

In [ ]:
from pathlib import Path

from src.hwp_to_faiss import (
    build_clean_docs_jsonl,
    build_chunk_docs_jsonl,
    build_faiss_from_chunk_jsonl,
)

# 경로(프로젝트 루트 기준)
ROOT = Path(r"C:\Users\최무영\Desktop\코드잇 스프린트\Team2-RAG-Project")

RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "out"
TMP_ROOT = ROOT / "tmp_hwp5html"
OUT_MD_DIR = OUT_DIR / "hwp_md"

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_MD_DIR.mkdir(parents=True, exist_ok=True)
TMP_ROOT.mkdir(parents=True, exist_ok=True)

# 1단 산출물(이미 있으면 재사용 가능)
CLEAN_DOCS_JSONL = OUT_DIR / "cleaned_hwp_docs.jsonl"

# chunk 파라미터
CHUNK_SIZE = 600
OVERLAP = 100
CHUNK_DOCS_JSONL = OUT_DIR / f"chunks_hwp_cs{CHUNK_SIZE}_ov{OVERLAP}.jsonl"
FAISS_DIR = OUT_DIR / f"faiss_hwp_cs{CHUNK_SIZE}_ov{OVERLAP}"

# (필수) .env 로딩 + 키 체크
from dotenv import load_dotenv
import os
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다(.env/환경변수 확인)"

# (선택) 이미 clean_jsonl이 있으면 1단은 스킵
if not CLEAN_DOCS_JSONL.exists():
    clean_report = build_clean_docs_jsonl(
        raw_dir=RAW_DIR,
        tmp_root=TMP_ROOT,
        out_md_dir=OUT_MD_DIR,
        clean_jsonl=CLEAN_DOCS_JSONL,
        cleanup_tmp=True,   # tmp 정리 토글
    )
    print("✅ 1단 리포트:", clean_report)
else:
    print("ℹ️ 1단 스킵: 기존 clean_jsonl 사용:", CLEAN_DOCS_JSONL)

# 2단(청킹)
chunk_report = build_chunk_docs_jsonl(
    clean_jsonl=CLEAN_DOCS_JSONL,
    chunk_jsonl=CHUNK_DOCS_JSONL,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
)
print("✅ 2단 리포트:", chunk_report)

# 3단(FAISS)
saved_faiss = build_faiss_from_chunk_jsonl(
    chunk_jsonl=CHUNK_DOCS_JSONL,
    faiss_dir=FAISS_DIR,
    embed_model="text-embedding-3-small",
    batch_size=100,
)
print("✅ 3단 FAISS:", saved_faiss)


ℹ️ 1단 스킵: 기존 clean_jsonl 사용: out\cleaned_hwp_docs.jsonl


(2) CLEAN -> CHUNKS cs=600, ov=100: 94it [00:00, 175.26it/s]


✅ 2단 리포트: out\chunks_hwp_cs600_ov100.report.csv


(3) CHUNKS -> FAISS (embed): 18740it [02:00, 155.54it/s]


✅ 3단 FAISS: out\faiss_hwp_cs600_ov100


## 2. pdf to faiss

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

# 0) 키 로딩
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다(.env/환경변수 확인)"

# 1) 함수 import
from src.pdf_to_faiss import (
    build_clean_pdf_jsonl,
    build_pdf_chunk_jsonl,
    build_faiss_from_chunk_jsonl,
    merge_chunk_jsonls,
)

# 2) 경로
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PDF_JSONL = OUT_DIR / "cleaned_pdf_docs.jsonl"

# 3) 청킹 파라미터(원하는 값으로 변경)
CHUNK_SIZE = 600
OVERLAP = 100

CHUNK_PDF_JSONL = OUT_DIR / f"chunks_pdf_cs{CHUNK_SIZE}_ov{OVERLAP}.jsonl"
FAISS_PDF_DIR   = OUT_DIR / f"faiss_pdf_cs{CHUNK_SIZE}_ov{OVERLAP}"

# 4) (1) PDF clean: 이미 있으면 스킵(원하면 조건 제거)
if not CLEAN_PDF_JSONL.exists():
    pdf_clean_report = build_clean_pdf_jsonl(RAW_DIR, CLEAN_PDF_JSONL, OUT_DIR)
    print("✅ PDF 1단 리포트:", pdf_clean_report)
else:
    print("ℹ️ PDF 1단 스킵: 기존 clean jsonl 사용:", CLEAN_PDF_JSONL)

# 5) (2) PDF chunk
pdf_chunk_report = build_pdf_chunk_jsonl(CLEAN_PDF_JSONL, CHUNK_PDF_JSONL, CHUNK_SIZE, OVERLAP)
print("✅ PDF 2단 리포트:", pdf_chunk_report)

# 6) (3) PDF FAISS
saved_pdf_faiss = build_faiss_from_chunk_jsonl(
    chunk_jsonl=CHUNK_PDF_JSONL,
    faiss_dir=FAISS_PDF_DIR,
    embed_model="text-embedding-3-small",
    batch_size=100,
)
print("✅ PDF 3단 FAISS:", saved_pdf_faiss)


ℹ️ PDF 1단 스킵: 기존 clean jsonl 사용: out\cleaned_pdf_docs.jsonl


(PDF-2) CLEAN -> CHUNKS cs=600, ov=100: 4it [00:00, 78.34it/s]


✅ PDF 2단 리포트: out\chunks_pdf_cs600_ov100.report.csv


(PDF-3) CHUNKS -> FAISS (embed): 1462it [00:07, 184.83it/s]


✅ PDF 3단 FAISS: out\faiss_pdf_cs600_ov100


## 3. faiss merge

In [3]:
from src.pdf_to_faiss import build_faiss_from_chunk_jsonl, merge_chunk_jsonls

CHUNK_HWP_JSONL = OUT_DIR / f"chunks_hwp_cs{CHUNK_SIZE}_ov{OVERLAP}.jsonl"

MERGED_CHUNK_JSONL = OUT_DIR / f"chunks_merged_hwp_pdf_cs{CHUNK_SIZE}_ov{OVERLAP}.jsonl"
FAISS_MERGED_DIR   = OUT_DIR / f"faiss_merged_hwp_pdf_cs{CHUNK_SIZE}_ov{OVERLAP}"

# 1) merged chunk jsonl 생성(덮어쓰기 주의: 이미 있으면 지우거나 다른 이름으로)
total_lines = merge_chunk_jsonls(MERGED_CHUNK_JSONL, [CHUNK_HWP_JSONL, CHUNK_PDF_JSONL])
print("✅ merged jsonl:", MERGED_CHUNK_JSONL, "lines=", total_lines)

# 2) merged FAISS 생성
saved_merged_faiss = build_faiss_from_chunk_jsonl(
    chunk_jsonl=MERGED_CHUNK_JSONL,
    faiss_dir=FAISS_MERGED_DIR,
    embed_model="text-embedding-3-small",
    batch_size=100,
)
print("✅ merged FAISS:", saved_merged_faiss)


✅ merged jsonl: out\chunks_merged_hwp_pdf_cs600_ov100.jsonl lines= 20202


(PDF-3) CHUNKS -> FAISS (embed): 20202it [02:08, 156.69it/s]


✅ merged FAISS: out\faiss_merged_hwp_pdf_cs600_ov100
